<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 UniGenBench Reproduction with Cosmos Framework

This notebook walks through generating the [UniGenBench](https://arxiv.org/abs/2510.18701) image-generation benchmark with [Cosmos3-Super-Text2Image](https://huggingface.co/nvidia/Cosmos3-Super-Text2Image) using the native Cosmos Framework PyTorch entrypoint:

```bash
python -m cosmos_framework.scripts.inference
```

UniGenBench is an **Text-to-Image (T2I)** benchmark: each case contains one text prompt and multiple testing points, and the model generates the designated image and a VLM will be utilized to exam the image against all testing points.

- **600+570=1170 testing cases** split into 600 original common domain cases and 570 addtional physical domain cases..

## Prerequisites

- Linux machine with NVIDIA GPU access (default recipe uses 4 GPUs).
- Model access on Hugging Face. Either run `uvx hf@latest auth login` or set `HF_TOKEN` in the environment.
- `uv >= 0.11.3` installed (https://docs.astral.sh/uv/getting-started/installation/).
- `git-lfs` on PATH. The RBench dataset (~22 GB) is downloaded by cloning the Hugging Face dataset repo, which stores images and checkpoints with Git LFS.
- Cache/output paths with enough disk space.

> **Headless servers:** if you see `libxcb.so.1: cannot open shared object file` when importing the model, install the system graphics libraries:
> ```bash
> apt-get install -y libxcb1 libgl1 libglib2.0-0
> ```

## 1. Configure Paths and Environment

All paths default to sensible locations under this `cosmos` checkout. Override any of them by exporting before launching the notebook:

```bash
export COSMOS3_REPO=/path/to/cosmos-framework
export COSMOS3_UV_GROUP=cu130-train   # or cu128-train
export UV_PROJECT_ENVIRONMENT=/path/to/large/uv/venvs/cosmos3-unigenbench
export COSMOS3_NUM_GPUS=4
export HF_HOME=/path/to/large/huggingface/cache
export CUDA_VISIBLE_DEVICES=0,1,2,3
export UNIGENBENCH_OUTPUT_ROOT=/path/to/unigenbench/outputs
```

In [ ]:
from pathlib import Path
import sys
import os
import os.path as osp
import socket
import json


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def free_local_port() -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return str(sock.getsockname()[1])


def default_framework_repo(root: Path) -> Path:
    for candidate in (root / "packages" / "cosmos-framework", root / "packages" / "cosmos3"):
        if (candidate / "pyproject.toml").exists() and (candidate / "cosmos_framework").exists():
            return candidate
    return root / "packages" / "cosmos-framework"


COSMOS3_ROOT = str(find_repo_root(Path.cwd().resolve()))
COSMOS3_REPO = os.environ.get("COSMOS3_REPO", default_framework_repo(COSMOS3_ROOT))
COSMOS3_GIT_URL = os.environ.get("COSMOS3_GIT_URL", "git@github.com:NVIDIA/cosmos-framework.git")
COSMOS3_UV_GROUP = os.environ.get("COSMOS3_UV_GROUP", "cu130-train")
COSMOS3_UV_ENV = os.environ.get("UV_PROJECT_ENVIRONMENT", osp.join(COSMOS3_REPO, ".venv"))
COSMOS3_NUM_GPUS = os.environ.get("COSMOS3_NUM_GPUS", "4")
COSMOS3_MASTER_ADDR = os.environ.get("COSMOS3_MASTER_ADDR", "127.0.0.1")
COSMOS3_MASTER_PORT = os.environ.get("COSMOS3_MASTER_PORT", free_local_port())

CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1,2,3")

UNIGENBENCH_JUDGE_GATEWAY_URL = ""
UNIGENBENCH_JUDGE_GATEWAY_API = ""
UNIGENBENCH_JUDGE_MODELSTR = ""

UNIGENBENCH_NOTEBOOK_ROOT = osp.join(COSMOS3_ROOT, "evaluation", "unigenbench")
UNIGENBENCH_PROMPT = osp.join(UNIGENBENCH_NOTEBOOK_ROOT, "assets", "unigenbench_prompt.json")
UNIGENBENCH_OUTPUT_ROOT = os.environ.get("UNIGENBENCH_OUTPUT_ROOT", osp.join(UNIGENBENCH_NOTEBOOK_ROOT, "outputs", "unigenbench_result.json"))
UNIGENBENCH_IMAGE = osp.join(UNIGENBENCH_OUTPUT_ROOT, "image")
UNIGENBENCH_SCORE = osp.join(UNIGENBENCH_OUTPUT_ROOT, "unigenbench_result.json")

CHECKPOINT = "Cosmos3-Super-Text2Image"

for key, value in [
    ("COSMOS3_ROOT"    , COSMOS3_ROOT),
    ("COSMOS3_REPO"    , COSMOS3_REPO),
    ("COSMOS3_UV_ENV"  , COSMOS3_UV_ENV),
    ("COSMOS3_NUM_GPUS", COSMOS3_NUM_GPUS),
    ("CUDA_VISIBLE_DEVICES", CUDA_VISIBLE_DEVICES),
    ("UNIGENBENCH_JUDGE_GATEWAY_URL", UNIGENBENCH_JUDGE_GATEWAY_URL),
    ("UNIGENBENCH_JUDGE_GATEWAY_API", UNIGENBENCH_JUDGE_GATEWAY_API),
    ("UNIGENBENCH_JUDGE_MODELSTR"   , UNIGENBENCH_JUDGE_MODELSTR),
    ("UNIGENBENCH_PROMPT"           , UNIGENBENCH_PROMPT),
    ("UNIGENBENCH_OUTPUT_ROOT"      , UNIGENBENCH_OUTPUT_ROOT),
    ("CHECKPOINT", CHECKPOINT),
]:
    print(f"{key}={value}")

# Export for the %%bash cells below.
os.environ["COSMOS3_REPO"]     = COSMOS3_REPO
os.environ["COSMOS3_GIT_URL"]  = COSMOS3_GIT_URL
os.environ["COSMOS3_UV_GROUP"] = COSMOS3_UV_GROUP
os.environ["COSMOS3_UV_ENV"]   = COSMOS3_UV_ENV
os.environ["UV_PROJECT_ENVIRONMENT"] = COSMOS3_UV_ENV
os.environ["COSMOS3_NUM_GPUS"] = COSMOS3_NUM_GPUS
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

os.environ["UNIGENBENCH_JUDGE_GATEWAY_URL"] = UNIGENBENCH_JUDGE_GATEWAY_URL
os.environ["UNIGENBENCH_JUDGE_GATEWAY_API"] = UNIGENBENCH_JUDGE_GATEWAY_API
os.environ["UNIGENBENCH_JUDGE_MODELSTR"]    = UNIGENBENCH_JUDGE_MODELSTR
os.environ["UNIGENBENCH_PROMPT"]            = UNIGENBENCH_PROMPT
os.environ["UNIGENBENCH_OUTPUT_ROOT"]       = UNIGENBENCH_OUTPUT_ROOT
os.environ["UNIGENBENCH_IMAGE"]             = UNIGENBENCH_IMAGE
os.environ["UNIGENBENCH_SCORE"]             = UNIGENBENCH_SCORE

os.environ["CHECKPOINT"] = CHECKPOINT
os.environ["COSMOS3_MASTER_ADDR"] = COSMOS3_MASTER_ADDR
os.environ["COSMOS3_MASTER_PORT"] = COSMOS3_MASTER_PORT

## 2. Clone or Reuse Cosmos Framework

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -f "$COSMOS3_REPO/pyproject.toml" ] && [ -d "$COSMOS3_REPO/cosmos_framework" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
elif [ -e "$COSMOS3_REPO" ]; then
  echo "COSMOS3_REPO exists but is not a Cosmos Framework checkout: $COSMOS3_REPO"
  exit 1
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

cd "$COSMOS3_REPO"
git status --short --branch
git remote -v

## 3. Install Native PyTorch Dependencies

Installs framework dependencies with the requested CUDA group (default `cu130-train`).

In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo "uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/"
  exit 1
fi

export GIT_LFS_SKIP_SMUDGE=1
cd "$COSMOS3_REPO"
export UV_PROJECT_ENVIRONMENT="${UV_PROJECT_ENVIRONMENT:-$COSMOS3_UV_ENV}"
echo "Using UV_PROJECT_ENVIRONMENT=$UV_PROJECT_ENVIRONMENT"
uv sync --all-extras --group="$COSMOS3_UV_GROUP"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "uv sync completed, but expected Python is missing: $COSMOS3_UV_ENV/bin/python"
  exit 1
fi

## 4. Verify GPU and Python Environment

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "Missing $COSMOS3_UV_ENV/bin/python"
  echo "Run the Install Native PyTorch Dependencies cell first."
  exit 1
fi
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" "$COSMOS3_UV_ENV/bin/python" - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"device {index}:", torch.cuda.get_device_name(index))
PY

## 5. Helper Functions

**T2I recipe**

- **Conditioning Text Prompt**: the JSON upsampled text prompt.
- **Negative prompt**:
```PYTHON
negative_prompt = (
    "The image shows ugly scenes, motion blur, over-saturation, "
    "shaky footage, low resolution, grainy texture, pixelated artifacts, "
    "poorly lit areas, underexposed and overexposed scenes, "
    "poor color balance, washed out colors, choppy sequences, jerky movements, "
    "artifacting, color banding, outdated special effects, fake elements, "
    "unconvincing visuals, poorly edited content, visual noise. "
    "Overall, the image is of poor quality."
)
```
- **Best Resolution** by aspect ratio:
  | Aspect ratio | Width × Height |
  |:---:|:---:|
  | 1:1  | 1024 × 1024 |
  | 4:3  | 1184 × 880  |
  | 3:4  | 880 × 1184  |
  | 16:9 | 1360 × 768  |
  | 9:16 | 768 × 1360  | 
- **Sampling**: `num_steps=50`, `guidance=4.0`, `shift=3.0`, `guidance_interval=[400,1000]` `fps=rand(16-to-32) # doesn't matter much`, `num_frames=1` (other sampler settings left at framework defaults).

Other Helpers:

In [ ]:
T2I_NUM_FRAMES = 1
T2I_FPS = 24 # random.randint(16, 32)
T2I_RESOLUTION = "768" # 1024 x 1024
T2I_ASPECT_RATIO = "1,1"
T2I_NUM_STEPS = 50
T2I_GUIDANCE = 4.0
T2I_SHIFT = 3.0
T2I_GUIDANCE_INTERVAL = [400, 1000]
T2I_START_SEED = 0
T2I_NEGATIVE_PROMPT = (
    "The image shows ugly scenes, motion blur, over-saturation, "
    "shaky footage, low resolution, grainy texture, pixelated artifacts, "
    "poorly lit areas, underexposed and overexposed scenes, "
    "poor color balance, washed out colors, choppy sequences, jerky movements, "
    "artifacting, color banding, outdated special effects, fake elements, "
    "unconvincing visuals, poorly edited content, visual noise. "
    "Overall, the image is of poor quality."
)


with open(UNIGENBENCH_PROMPT, "r") as f:
    PROMPTS = json.load(f)["benchmark"]
    PROMPTS = {p['orig0'] : p for p in PROMPTS}


def build_row(case_id: str, seed: int) -> dict:
    entry = PROMPTS[case_id]
    return {
        "name": case_id,
        "model_mode": "text2image",
        "prompt": json.dumps(entry["upsampled_prompt"], indent=4, ensure_ascii=True),
        "negative_prompt": T2I_NEGATIVE_PROMPT,

        "aspect_ratio": T2I_ASPECT_RATIO,
        "resolution": T2I_RESOLUTION,
        "fps": T2I_FPS,
        "num_frames": T2I_NUM_FRAMES,

        "num_steps": T2I_NUM_STEPS,
        "shift": T2I_SHIFT,
        "guidance_interval": T2I_GUIDANCE_INTERVAL,
        "guidance": T2I_GUIDANCE,
        "seed": seed,
        "num_outputs": 1,
    }


def build_input_jsonl(input_file, id_list) -> None:
    for id in id_list:
        Path(input_file).write_text(json.dumps(build_row(id)) + "\n")


## 6. T2I Inference

We use the **latency** parallelism preset with context-parallel sharding across all visible GPUs (`--cp-size=$COSMOS3_NUM_GPUS`). 

In [ ]:
output_dir = UNIGENBENCH_IMAGE
os.makedirs(output_dir, exist_ok=True)

id_list = sorted(list(PROMPTS.keys()))
debug_limit_n = 10
if debug_limit_n > 0:
    id_list = id_list[:debug_limit_n]

input_jsonl = osp.join(UNIGENBENCH_OUTPUT_ROOT, "input.jsonl")
build_input_jsonl(input_jsonl, id_list)

os.environ["T2I_INPUT"]      = input_jsonl
os.environ["T2I_OUTPUT_DIR"] = output_dir

print(f"T2I_INPUT={input_jsonl}")
print(f"T2I_OUTPUT_DIR={output_dir}")


In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 \
  --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" \
  --cfgp-size=1 \
  -i "$T2I_INPUT" \
  -o "$T2I_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails


## 7. Get T2I UniGenBench Score

In [ ]:
import subprocess

scorer_path = osp.join(COSMOS3_ROOT, "evaluation", "cosmos3", "generator", "unigenbench", "ugb_scorer.py")
python_bin  = osp.join(COSMOS3_UV_ENV, "bin", "python")

subprocess.run(
    [
        python_bin,
        scorer_path,
        "--image_folder",          UNIGENBENCH_IMAGE,
        "--benchmark_prompt_json", UNIGENBENCH_PROMPT,
        "--output_score_json",     UNIGENBENCH_SCORE,
        "--gateway_url",           UNIGENBENCH_JUDGE_GATEWAY_URL,
        "--gateway_api",           UNIGENBENCH_JUDGE_GATEWAY_API,
        "--modelstr",              UNIGENBENCH_JUDGE_MODELSTR,
    ],
    check=True,
)
